# 📈 Notebook 04 — Model Evaluation & SHAP Analysis
**FMCG Promotional Analytics | Portfolio Project**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Panosemmanouilidis2/fmcg-ds-technical-portfolio/blob/main/promotional-analytics/notebooks/04_model_evaluation_shap.ipynb)

This notebook evaluates the trained XGBoost model, analyses prediction errors, runs SHAP explainability, and derives ROI insights from model predictions.

| | |
|---|---|
| Input | Runs Notebook 02 pipeline inline — fully self-contained |
| Explainability | SHAP (SHapley Additive exPlanations) |
| Business output | Predicted ROI classification per promotion |

> **Note:** This notebook re-runs feature engineering and training internally so it works standalone in Colab without needing Notebook 02's saved files.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 1 — INSTALL & IMPORT
# ══════════════════════════════════════════════════════════════════
!pip install lightgbm shap --quiet

import pandas as pd
import numpy as np
import json, pickle, os, warnings
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import shap
warnings.filterwarnings('ignore')

os.makedirs('results', exist_ok=True)
print("✅ Imports OK")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 2 — LOAD & PREPARE DATA  (self-contained — loads from GitHub)
# Mirrors the feature engineering from Notebook 02 so this notebook
# runs standalone without requiring saved files from Notebook 02.
# ══════════════════════════════════════════════════════════════════
CSV_URL = "https://raw.githubusercontent.com/Panosemmanouilidis2/fmcg-ds-technical-portfolio/main/promotional-analytics/notebooks/sample_synthetic.csv"
TARGET  = 'ActualSellOutVolume'
LEAKAGE = [TARGET, 'PlanningAccuracyRatio']

df = pd.read_csv(CSV_URL)
df['Market'] = df['Market'].map({'Western Europe':'Market A','Southeast Asia':'Market B'})

df_enc = df.copy()
df_enc = pd.get_dummies(df_enc, columns=['Market','PromoMechanic','PromoFeature','Division'],
                        drop_first=False)
for col in ['Customer','Category','PromotionMonth']:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

df_enc['LogTarget'] = np.log1p(df_enc[TARGET])
df_sorted = df_enc.sort_values('InstoreStartDate_Month').reset_index(drop=True)
feature_cols = [c for c in df_sorted.columns if c not in LEAKAGE + ['LogTarget']]

X = df_sorted[feature_cols]
y = df_sorted['LogTarget']
split_idx   = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {len(X_train):,}  Test: {len(X_test):,}  Features: {len(feature_cols)}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 3 — TRAIN TUNED XGBOOST
# Uses best params from Notebook 02 tuning session.
# Re-runs here so this notebook is fully self-contained.
# ══════════════════════════════════════════════════════════════════
param_grid = {
    'n_estimators'    : [200, 400, 600],
    'max_depth'       : [3, 5, 7],
    'learning_rate'   : [0.01, 0.05, 0.1],
    'subsample'       : [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'reg_alpha'       : [0, 0.1, 1.0],
    'reg_lambda'      : [1.0, 5.0, 10.0],
}
sample_size = min(2000, len(X_train))
search = RandomizedSearchCV(
    xgb.XGBRegressor(objective='reg:squarederror', verbosity=0, random_state=42),
    param_grid, n_iter=30, cv=3, scoring='r2', random_state=42, n_jobs=-1
)
search.fit(X_train.iloc[:sample_size], y_train.iloc[:sample_size])

xgb_tuned = xgb.XGBRegressor(**search.best_params_,
                               objective='reg:squarederror',
                               verbosity=0, random_state=42)
xgb_tuned.fit(X_train, y_train)
y_pred = xgb_tuned.predict(X_test)
print(f"✅ Model trained  R²={r2_score(y_test, y_pred):.4f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 4 — MODEL PERFORMANCE METRICS
# R²   : proportion of variance explained (1.0 = perfect)
# MAE  : average absolute error on log scale
# RMSE : root mean squared error — penalises large errors more
# MAPE : mean absolute % error on original unit scale
# ══════════════════════════════════════════════════════════════════
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

act_orig  = np.expm1(y_test.values)
pred_orig = np.expm1(y_pred)
mape = np.mean(np.abs((act_orig - pred_orig) / act_orig.clip(min=1))) * 100

print("=== MODEL EVALUATION (log scale) ===")
print(f"  R²   : {r2:.4f}  ({'Excellent' if r2>0.75 else 'Good' if r2>0.6 else 'Fair'})")
print(f"  MAE  : {mae:.4f}")
print(f"  RMSE : {rmse:.4f}")
print(f"\n=== ORIGINAL UNIT SCALE ===")
print(f"  Mean Actual    : {act_orig.mean():>12,.0f} units")
print(f"  Mean Predicted : {pred_orig.mean():>12,.0f} units")
print(f"  MAPE           : {mape:.1f}%")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 5 — PREDICTED vs ACTUAL SCATTER
# Points along the diagonal = perfect predictions.
# Scatter above the line = model under-predicted.
# Scatter below the line = model over-predicted.
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.25, s=8, color='#378ADD')
lims = [y_test.min(), y_test.max()]
ax.plot(lims, lims, 'r--', linewidth=1.2, label='Perfect prediction')
ax.set_xlabel('Actual (log scale)')
ax.set_ylabel('Predicted (log scale)')
ax.set_title(f'Predicted vs Actual (log)  R²={r2:.3f}', fontsize=12)
ax.legend()

ax = axes[1]
clip = 200_000
ax.scatter(act_orig.clip(max=clip), pred_orig.clip(max=clip),
           alpha=0.25, s=8, color='#1D9E75')
ax.plot([0, clip], [0, clip], 'r--', linewidth=1.2, label='Perfect prediction')
ax.set_xlabel('Actual Units (clipped at 200k)')
ax.set_ylabel('Predicted Units (clipped at 200k)')
ax.set_title('Predicted vs Actual (units)', fontsize=12)
ax.legend()

plt.tight_layout()
plt.savefig('results/04_predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 6 — RESIDUAL ANALYSIS
# Residuals = Predicted − Actual on log scale.
# A good model: residuals centred on 0, no fan pattern.
# Fan pattern (heteroscedasticity): error grows with volume.
# ══════════════════════════════════════════════════════════════════
residuals = y_pred - y_test.values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals, bins=60, color='#378ADD', edgecolor='white', linewidth=0.4)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.2)
axes[0].set_title(f'Residual Distribution  (mean={residuals.mean():.3f})', fontsize=12)
axes[0].set_xlabel('Predicted − Actual (log scale)')
axes[0].set_ylabel('Count')

axes[1].scatter(y_test, residuals, alpha=0.2, s=8, color='#E87B2E')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.2)
axes[1].set_title('Residuals vs Actual Volume', fontsize=12)
axes[1].set_xlabel('Actual (log scale)')
axes[1].set_ylabel('Residual')

plt.tight_layout()
plt.savefig('results/04_residuals.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 7 — SHAP FEATURE IMPORTANCE
# SHAP = SHapley Additive exPlanations.
# For every prediction, SHAP shows how much each feature pushed
# the result up or down from the average prediction.
# Unlike standard feature importance, SHAP shows direction too.
# ══════════════════════════════════════════════════════════════════
sample_idx  = np.random.RandomState(42).choice(len(X_test),
                size=min(500, len(X_test)), replace=False)
X_sample    = X_test.iloc[sample_idx]

explainer   = shap.TreeExplainer(xgb_tuned)
shap_values = explainer.shap_values(X_sample)
shap_df     = pd.DataFrame(shap_values, columns=feature_cols)

mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False).head(15)

print("Top 10 features by mean |SHAP|:")
for feat, val in mean_abs_shap.head(10).items():
    print(f"  {feat:<45} {val:.4f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 8 — SHAP TOP FEATURES BAR CHART
# ══════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 6))
mean_abs_shap[::-1].plot(kind='barh', ax=ax, color='#378ADD', edgecolor='white')
ax.set_title('Top 15 Features by Mean |SHAP| Value', fontsize=13)
ax.set_xlabel('Mean |SHAP| (average impact on log sell-out prediction)')
plt.tight_layout()
plt.savefig('results/04_shap_top_features.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 9 — SHAP SUMMARY (BEE SWARM) PLOT
# Each dot = one prediction.
# Colour = feature value (blue = low, red = high).
# x-axis position = SHAP contribution
#   (right = pushed prediction up, left = pushed it down).
# ══════════════════════════════════════════════════════════════════
top_features = mean_abs_shap.head(15).index.tolist()
top_indices  = [feature_cols.index(f) for f in top_features]

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values[:, top_indices],
                  X_sample[top_features],
                  feature_names=top_features,
                  show=False)
plt.title('SHAP Summary — Top 15 Features', fontsize=13)
plt.tight_layout()
plt.savefig('results/04_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 10 — ROI CLASSIFICATION
# Classify each test promo as ROI-positive or ROI-negative using
# the model's predicted sell-out against the planned baseline.
# A promo is ROI-positive when predicted sell-out exceeds baseline
# AND the planned ROI proxy indicates positive margin contribution.
# ══════════════════════════════════════════════════════════════════
eval_df = X_test.copy()
eval_df['ActualSellOut']    = act_orig
eval_df['PredictedSellOut'] = pred_orig

if 'PlannedROI_proxy' in eval_df.columns and 'PlannedBaselineVolume' in eval_df.columns:
    eval_df['ROI_positive'] = (
        (eval_df['PredictedSellOut'] > eval_df['PlannedBaselineVolume']) &
        (eval_df['PlannedROI_proxy'] > 0)
    ).astype(int)
else:
    eval_df['ROI_positive'] = (
        eval_df['PredictedSellOut'] > eval_df['ActualSellOut'].median()
    ).astype(int)

roi_pct = eval_df['ROI_positive'].mean() * 100
print(f"=== ROI CLASSIFICATION ===")
print(f"  ROI-positive : {eval_df['ROI_positive'].sum():,}  ({roi_pct:.1f}%)")
print(f"  ROI-negative : {(~eval_df['ROI_positive'].astype(bool)).sum():,}  ({100-roi_pct:.1f}%)")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 11 — MAPE BY PROMO MECHANIC
# Which mechanics does the model predict most accurately?
# ══════════════════════════════════════════════════════════════════
mech_cols = [c for c in eval_df.columns if c.startswith('PromoMechanic_')]

if mech_cols:
    rows = []
    for col in mech_cols:
        mask = eval_df[col] == 1
        if mask.sum() >= 10:
            sub = eval_df[mask]
            mape_m = np.mean(np.abs(
                (sub['ActualSellOut'] - sub['PredictedSellOut']) /
                sub['ActualSellOut'].clip(min=1)
            )) * 100
            rows.append({'Mechanic': col.replace('PromoMechanic_',''),
                         'MAPE_%': round(mape_m, 1), 'Count': mask.sum()})
    mech_df = pd.DataFrame(rows).sort_values('MAPE_%')
    print("=== MAPE BY PROMO MECHANIC ===")
    print(mech_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#1D9E75' if v < 50 else '#D85A30' for v in mech_df['MAPE_%']]
    ax.barh(mech_df['Mechanic'], mech_df['MAPE_%'], color=colors, edgecolor='white')
    ax.axvline(mech_df['MAPE_%'].median(), color='black', linestyle='--',
               linewidth=1.2, label='Median MAPE')
    ax.set_title('Prediction MAPE by Promo Mechanic', fontsize=13)
    ax.set_xlabel('MAPE %  (lower = more accurate)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('results/04_mape_by_mechanic.png', dpi=150, bbox_inches='tight')
    plt.show()


## 📝 Evaluation Summary

| Metric | Description |
|---|---|
| R² | Proportion of variance explained — see Cell 4 |
| Top SHAP driver | Planned baseline volume — high baseline = higher sell-out |
| ROI-positive rate | See Cell 10 output |
| Least predictable | Pipe Fill and Other (structural noise in data) |

**Pipeline complete.** Model trained → deployed → evaluated → explained.
